In [10]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [11]:
from src import config

In [12]:
from datetime import datetime, timedelta

import pandas as pd

current_data = pd.to_datetime(datetime.utcnow()).floor("h")
print(f"{current_data=}")

# we fetch raw data for the last 28 days, to add redundency to our data pipeline
fetch_data_to = current_data
fetch_data_from = current_data - timedelta(days=28)

current_data=Timestamp('2026-06-24 08:00:00')


In [13]:
from src.data import load_row_data

def fetch_batch_raw_data(from_date: datetime, to_date: datetime) -> pd.DataFrame:
    """
    Simulate production date by sampling historical data from 52 weeks ago .
    """
    from_date_ = from_date - timedelta(days=7*52)
    to_date_ = to_date - timedelta(days=7*52)

    # downalod 2 files from website
    rides = load_row_data(year=from_date_.year,months=from_date_.month)
    rides = rides[rides.pickup_datetime >= from_date_]
    rides_2 = load_row_data(year=to_date_.year, months=to_date_.month)
    rides_2 = rides_2[rides_2.pickup_datetime < to_date_]

    rides = pd.concat([rides, rides_2])

    # Shift the data to pretend this is recrent data
    rides['pickup_datetime'] += timedelta(days=7*52)

    rides.sort_values(
        by=['pickup_location_id','pickup_datetime'],
        inplace=True
    )

    return rides

In [14]:
rides = fetch_batch_raw_data(
    from_date=fetch_data_from,
    to_date=fetch_data_to
)
print(type(rides))

file 2025-05 was already in local storage
file 2025-06 was already in local storage
<class 'pandas.core.frame.DataFrame'>


In [15]:
print(type(rides))

<class 'pandas.core.frame.DataFrame'>


In [16]:
from src.data import tranfrom_row_data_into_ts_data
ts_data = tranfrom_row_data_into_ts_data(rides)

100%|██████████| 259/259 [00:00<00:00, 406.71it/s]


In [17]:
import hopsworks

# connect to the project
project = hopsworks.login(
    project=config.HOPSWORKS_PROJECT_NAME,
    api_key_value=config.HOPSWORKS_API_KEY,
    cert_folder="./hopsworks-certs",
)

# connect to feature store
feature_store = project.get_feature_store()

# connect to feature group 
feature_group = feature_store.get_or_create_feature_group(
    name=config.FEATURE_GROUP_NAME,
    version=config.FEATURE_GROUP_VERSION,
    description="Time-series data at hourly frequency",
    primary_key=["pickup_location_id", "pickup_hour"],
    event_time="pickup_hour",
    time_travel_format="HUDI",
)

2026-06-24 11:11:41,215 INFO: Closing external client and cleaning up certificates.
2026-06-24 11:11:41,220 INFO: Connection closed.
2026-06-24 11:11:41,221 INFO: Initializing external client
2026-06-24 11:11:41,222 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-06-24 11:11:42,831 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/34960


In [18]:
feature_group.insert(ts_data, write_options={"wait_for_job": True})

Uploading Dataframe: 100.00% |██████████| Rows 174048/174048 | Elapsed Time: 01:06 | Remaining Time: 00:00


Launching job: time_series_hourly_feature_group_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/34960/jobs/named/time_series_hourly_feature_group_1_offline_fg_materialization/executions


2026-06-24 11:13:03,508 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-06-24 11:13:06,710 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-06-24 11:15:37,086 INFO: Waiting for execution to finish. Current state: AGGREGATING_LOGS. Final status: SUCCEEDED
2026-06-24 11:15:37,249 INFO: Waiting for log aggregation to finish.
2026-06-24 11:15:46,038 INFO: Execution finished successfully.


(Job('time_series_hourly_feature_group_1_offline_fg_materialization', 'SPARK'),
 None)

In [19]:
feature_group.read().head()

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (30.32s) 


,pickup_hour,rides,pickup_location_id
0,2025-03-07 21:00:00+00:00,78,233
1,2025-11-07 03:00:00+00:00,10,141
2,2025-08-16 14:00:00+00:00,1,136
3,2026-04-15 22:00:00+00:00,91,211
4,2025-12-30 09:00:00+00:00,0,128
